# Camada Gold — Mercado Financeiro e Criptomoedas

Este notebook constrói as tabelas analíticas do MVP a partir dos dados tratados na camada Silver.

O objetivo é preparar informações para análise de desempenho, risco, retorno real e relações entre criptomoedas, Ibovespa e indicadores macroeconômicos.

O período de observação compreende janeiro de 2021 a dezembro de 2025. As comparações principais utilizam frequência mensal para compatibilizar os diferentes calendários das fontes.

As criptomoedas são originalmente cotadas em USDT. As conversões para reais são aproximações que utilizam a PTAX USD/BRL e assumem paridade entre USDT e USD. Essa limitação é preservada na interpretação dos resultados.

Os dados analíticos são persistidos em tabelas Delta no schema `gold`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import types as T

catalogo = spark.sql(
    "SELECT current_catalog() AS catalogo"
).collect()[0]["catalogo"]

def nome_tabela(schema, tabela):
    return f"`{catalogo}`.`{schema}`.`{tabela}`"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalogo}`.`gold`
COMMENT 'Camada Gold do MVP: indicadores e tabelas analíticas'
""")

print("Catálogo:", catalogo)
print("Schema Gold preparado.")

## Leitura das tabelas Silver

Os dados são lidos diretamente das cinco tabelas Delta persistidas na camada Silver. Nenhuma API externa ou notebook de coleta é executado nesta etapa.

As transformações preservam as unidades e os calendários originais das fontes, realizando a compatibilização temporal apenas quando necessária para as análises.

In [0]:
criptos = spark.table(nome_tabela("silver", "criptomoedas"))
dolar = spark.table(nome_tabela("silver", "dolar"))
selic = spark.table(nome_tabela("silver", "selic"))
ipca = spark.table(nome_tabela("silver", "ipca"))
ibov = spark.table(nome_tabela("silver", "ibovespa"))

print("Tabelas Silver carregadas.")

## 1. Preços e retornos diários

A tabela diária reúne as criptomoedas e o Ibovespa em uma estrutura comum.

O retorno simples é calculado pela razão entre o fechamento atual e o fechamento da observação anterior, subtraída de 1.

As criptomoedas permanecem cotadas em USDT e o Ibovespa em pontos. Não são criados registros artificiais para datas sem negociação.

O retorno diário representa a variação entre observações consecutivas. No Ibovespa, por exemplo, o retorno de segunda-feira normalmente utiliza o fechamento da sexta-feira anterior.

In [0]:
precos_criptos = (
    criptos
    .select(
        "data",
        "ativo",
        F.lit("CRIPTO").alias("classe_ativo"),
        F.lit("USDT").alias("unidade_preco"),
        F.col("fechamento").alias("preco_fechamento")
    )
)

precos_ibov = (
    ibov
    .select(
        "data",
        "ativo",
        F.lit("INDICE").alias("classe_ativo"),
        F.lit("PONTOS").alias("unidade_preco"),
        F.col("fechamento").alias("preco_fechamento")
    )
)

janela_diaria = (
    Window
    .partitionBy("ativo")
    .orderBy("data")
)

gold_precos_diarios = (
    precos_criptos
    .unionByName(precos_ibov)
    .withColumn(
        "preco_anterior",
        F.lag("preco_fechamento").over(janela_diaria)
    )
    .withColumn(
        "retorno_diario",
        F.when(
            F.col("preco_anterior").isNotNull(),
            F.col("preco_fechamento") /
            F.col("preco_anterior") - 1
        )
    )
    .drop("preco_anterior")
)

## 2. Fechamentos e retornos mensais

Para cada ativo, é selecionado o último fechamento disponível em cada mês.

O retorno mensal é calculado entre fechamentos de meses consecutivos. Janeiro de 2021 é utilizado como referência inicial, pois não há dados de dezembro de 2020 para calcular seu retorno.

A frequência mensal permite comparar os cinco ativos respeitando seus diferentes calendários de negociação.

In [0]:
janela_mes = (
    Window
    .partitionBy("ativo", "mes_referencia")
    .orderBy(F.col("data").desc())
)

fechamentos_mensais = (
    gold_precos_diarios
    .withColumn(
        "mes_referencia",
        F.trunc("data", "month")
    )
    .withColumn(
        "ordem",
        F.row_number().over(janela_mes)
    )
    .filter(F.col("ordem") == 1)
    .select(
        "mes_referencia",
        F.col("data").alias("data_fechamento"),
        "ativo",
        "classe_ativo",
        "unidade_preco",
        "preco_fechamento"
    )
)

janela_ativo = (
    Window
    .partitionBy("ativo")
    .orderBy("mes_referencia")
)

gold_mercado_mensal = (
    fechamentos_mensais
    .withColumn(
        "preco_mes_anterior",
        F.lag("preco_fechamento").over(janela_ativo)
    )
    .withColumn(
        "retorno_mensal",
        F.when(
            F.col("preco_mes_anterior").isNotNull(),
            F.col("preco_fechamento") /
            F.col("preco_mes_anterior") - 1
        )
    )
    .drop("preco_mes_anterior")
)

## 3. Indicadores macroeconômicos mensais

A cotação do dólar é representada pela última PTAX de venda disponível no mês. A Meta Selic é representada pelo último valor observado no mês, mantendo a unidade percentual ao ano.

O IPCA permanece na frequência mensal e é convertido em fator de inflação para possibilitar o cálculo de retorno real.

As datas de referência dos indicadores não são interpretadas como datas de divulgação. As análises são retrospectivas e não constituem uma estratégia de previsão ou negociação.

In [0]:
janela_dolar = (
    Window
    .partitionBy("mes_referencia")
    .orderBy(F.col("data").desc())
)

dolar_mensal = (
    dolar
    .withColumn(
        "mes_referencia",
        F.trunc("data", "month")
    )
    .withColumn(
        "ordem",
        F.row_number().over(janela_dolar)
    )
    .filter(F.col("ordem") == 1)
    .select(
        "mes_referencia",
        F.col("data").alias("data_cotacao_dolar"),
        F.col("cotacao_venda").alias("usd_brl")
    )
)

janela_macro = Window.orderBy("mes_referencia")

dolar_mensal = (
    dolar_mensal
    .withColumn(
        "usd_brl_anterior",
        F.lag("usd_brl").over(janela_macro)
    )
    .withColumn(
        "variacao_dolar_mensal",
        F.when(
            F.col("usd_brl_anterior").isNotNull(),
            F.col("usd_brl") /
            F.col("usd_brl_anterior") - 1
        )
    )
    .drop("usd_brl_anterior")
)

In [0]:
janela_selic = (
    Window
    .partitionBy("mes_referencia")
    .orderBy(F.col("data").desc())
)

selic_mensal = (
    selic
    .withColumn(
        "mes_referencia",
        F.trunc("data", "month")
    )
    .withColumn(
        "ordem",
        F.row_number().over(janela_selic)
    )
    .filter(F.col("ordem") == 1)
    .select(
        "mes_referencia",
        "selic_meta_pct_aa"
    )
)

gold_macro_mensal = (
    ipca
    .select(
        "mes_referencia",
        "ipca_pct_mes"
    )
    .join(
        dolar_mensal,
        on="mes_referencia",
        how="left"
    )
    .join(
        selic_mensal,
        on="mes_referencia",
        how="left"
    )
    .withColumn(
        "fator_inflacao",
        1 + F.col("ipca_pct_mes") / 100
    )
    .withColumn(
        "selic_anterior",
        F.lag("selic_meta_pct_aa").over(janela_macro)
    )
    .withColumn(
        "variacao_selic_pp",
        F.col("selic_meta_pct_aa") -
        F.col("selic_anterior")
    )
    .drop("selic_anterior")
)

## 4. Integração mensal de mercado e macroeconomia

Os dados de mercado são associados aos indicadores macroeconômicos pelo mês de referência.

Para criptomoedas, é calculado um preço aproximado em reais utilizando a PTAX USD/BRL e assumindo paridade entre USDT e USD. A aproximação não representa uma cotação efetiva de negociação da stablecoin e não considera custos, spreads ou impostos.

O retorno do Ibovespa permanece baseado na variação do índice em pontos. O retorno real mensal é calculado pela razão entre o fator de retorno nominal em reais e o fator de inflação, subtraída de 1.

A primeira observação mensal de cada ativo permanece sem retorno, pois não existe fechamento anterior dentro da janela coletada.

In [0]:
gold_analise_mensal = (
    gold_mercado_mensal
    .join(
        gold_macro_mensal,
        on="mes_referencia",
        how="left"
    )
    .withColumn(
        "preco_aproximado_brl",
        F.when(
            F.col("classe_ativo") == "CRIPTO",
            F.col("preco_fechamento") *
            F.col("usd_brl")
        )
    )
)

janela_mensal = (
    Window
    .partitionBy("ativo")
    .orderBy("mes_referencia")
)

gold_analise_mensal = (
    gold_analise_mensal
    .withColumn(
        "preco_brl_anterior",
        F.lag("preco_aproximado_brl").over(janela_mensal)
    )
    .withColumn(
        "retorno_brl",
        F.when(
            F.col("classe_ativo") == "INDICE",
            F.col("retorno_mensal")
        ).otherwise(
            F.when(
                F.col("preco_brl_anterior").isNotNull(),
                F.col("preco_aproximado_brl") /
                F.col("preco_brl_anterior") - 1
            )
        )
    )
    .withColumn(
        "retorno_real_mensal",
        F.when(
            F.col("retorno_brl").isNotNull(),
            (1 + F.col("retorno_brl")) /
            F.col("fator_inflacao") - 1
        )
    )
    .drop("preco_brl_anterior")
)

## 5. Desempenho e risco dos ativos

Os indicadores de desempenho são calculados a partir dos retornos mensais em reais, considerando fevereiro de 2021 a dezembro de 2025.

São calculados retorno acumulado, retorno anualizado composto, volatilidade anualizada, relação retorno/volatilidade, retorno real acumulado e drawdown máximo.

A volatilidade utiliza o desvio-padrão amostral dos retornos mensais multiplicado pela raiz quadrada de 12. A relação retorno/volatilidade não corresponde ao índice de Sharpe, pois não desconta uma taxa livre de risco.

O drawdown é calculado sobre a trajetória acumulada iniciada em janeiro de 2021, sendo limitado ao período observado. Não representa o maior drawdown histórico de toda a existência dos ativos.

In [0]:
base_retornos = (
    gold_analise_mensal
    .filter(F.col("retorno_brl").isNotNull())
    .select(
        "mes_referencia",
        "ativo",
        "classe_ativo",
        "retorno_mensal",
        "retorno_brl",
        "retorno_real_mensal"
    )
)

gold_desempenho_ativos = (
    base_retornos
    .groupBy("ativo", "classe_ativo")
    .agg(
        F.count("retorno_brl").alias("meses_com_retorno"),
        F.min("mes_referencia").alias("primeiro_mes_retorno"),
        F.max("mes_referencia").alias("ultimo_mes_retorno"),
        F.avg("retorno_brl").alias("retorno_medio_mensal"),
        F.stddev_samp("retorno_brl").alias("volatilidade_mensal"),
        F.sum(
            F.log(1 + F.col("retorno_brl"))
        ).alias("soma_log_retorno"),
        F.sum(
            F.log(1 + F.col("retorno_real_mensal"))
        ).alias("soma_log_retorno_real")
    )
    .withColumn(
        "retorno_acumulado",
        F.exp(F.col("soma_log_retorno")) - 1
    )
    .withColumn(
        "retorno_anualizado",
        F.exp(
            F.col("soma_log_retorno") * 12 /
            F.col("meses_com_retorno")
        ) - 1
    )
    .withColumn(
        "volatilidade_anualizada",
        F.col("volatilidade_mensal") * F.sqrt(F.lit(12.0))
    )
    .withColumn(
        "relacao_retorno_volatilidade",
        F.when(
            F.col("volatilidade_anualizada") > 0,
            F.col("retorno_anualizado") /
            F.col("volatilidade_anualizada")
        )
    )
    .withColumn(
        "retorno_real_acumulado",
        F.exp(F.col("soma_log_retorno_real")) - 1
    )
    .drop("soma_log_retorno", "soma_log_retorno_real")
)

In [0]:
janela_acumulada = (
    Window
    .partitionBy("ativo")
    .orderBy("mes_referencia")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

base_drawdown = (
    base_retornos
    .withColumn(
        "indice_crescimento",
        F.exp(
            F.sum(
                F.log(1 + F.col("retorno_brl"))
            ).over(janela_acumulada)
        )
    )
    .withColumn(
        "pico_acumulado",
        F.greatest(
            F.lit(1.0),
            F.max("indice_crescimento").over(janela_acumulada)
        )
    )
    .withColumn(
        "drawdown",
        F.col("indice_crescimento") /
        F.col("pico_acumulado") - 1
    )
)

drawdown_ativos = (
    base_drawdown
    .groupBy("ativo")
    .agg(
        F.min("drawdown").alias("drawdown_maximo")
    )
)

gold_desempenho_ativos = (
    gold_desempenho_ativos
    .join(
        drawdown_ativos,
        on="ativo",
        how="left"
    )
)

display(
    gold_desempenho_ativos
    .orderBy(F.desc("retorno_acumulado"))
)

## 6. Volatilidade por ano

A volatilidade é calculada separadamente para cada ano, utilizando os retornos mensais disponíveis naquele período.

A anualização utiliza a raiz quadrada de 12. Para 2021, existem apenas 11 retornos mensais comparáveis, pois janeiro é o mês de referência inicial.

A análise permite identificar períodos de maior instabilidade e comparar o comportamento de risco dos ativos ao longo dos cinco anos.

In [0]:
gold_volatilidade_anual = (
    base_retornos
    .withColumn(
        "ano",
        F.year("mes_referencia")
    )
    .groupBy("ativo", "classe_ativo", "ano")
    .agg(
        F.count("retorno_brl").alias("meses_com_retorno"),
        F.avg("retorno_brl").alias("retorno_medio_mensal"),
        F.stddev_samp("retorno_brl").alias("volatilidade_mensal"),
        F.sum(
            F.log(1 + F.col("retorno_brl"))
        ).alias("soma_log_retorno")
    )
    .withColumn(
        "retorno_acumulado_ano",
        F.exp(F.col("soma_log_retorno")) - 1
    )
    .withColumn(
        "volatilidade_anualizada",
        F.col("volatilidade_mensal") * F.sqrt(F.lit(12.0))
    )
    .drop("soma_log_retorno")
)

display(
    gold_volatilidade_anual
    .orderBy("ano", "ativo")
)

## 7. Correlações entre ativos e indicadores

As correlações são calculadas utilizando observações mensais alinhadas pelo mês de referência.

São consideradas duas perspectivas para os ativos: retornos originais, que preservam as unidades de cotação, e retornos aproximados em reais, que permitem avaliar o efeito conjunto do ativo e da variação cambial.

As relações com o dólar são analisadas a partir da variação mensal da PTAX. A Meta Selic é analisada tanto pelo nível anual ao final do mês quanto pela variação em pontos percentuais.

A correlação de Pearson mede associação linear e não permite estabelecer causalidade. A interpretação considera a frequência, o tamanho da amostra e as limitações das séries.

In [0]:
ativos = [
    "BTCUSDT",
    "ETHUSDT",
    "SOLUSDT",
    "XRPUSDT",
    "IBOV"
]

retornos_brl_largos = (
    gold_analise_mensal
    .groupBy("mes_referencia")
    .pivot("ativo", ativos)
    .agg(F.first("retorno_brl"))
)

retornos_originais_largos = (
    gold_analise_mensal
    .groupBy("mes_referencia")
    .pivot("ativo", ativos)
    .agg(F.first("retorno_mensal"))
    .select(
        "mes_referencia",
        *[
            F.col(ativo).alias(f"{ativo}_ORIGINAL")
            for ativo in ativos
        ]
    )
)

base_correlacoes = (
    retornos_brl_largos
    .join(
        retornos_originais_largos,
        on="mes_referencia",
        how="inner"
    )
    .join(
        gold_macro_mensal.select(
            "mes_referencia",
            "variacao_dolar_mensal",
            "selic_meta_pct_aa",
            "variacao_selic_pp"
        ),
        on="mes_referencia",
        how="left"
    )
)

In [0]:
variaveis_brl = ativos + [
    "variacao_dolar_mensal",
    "selic_meta_pct_aa",
    "variacao_selic_pp"
]

variaveis_originais = [
    f"{ativo}_ORIGINAL"
    for ativo in ativos
] + [
    "variacao_dolar_mensal",
    "selic_meta_pct_aa",
    "variacao_selic_pp"
]

pares_correlacao = []

for perspectiva, variaveis in [
    ("RETORNO_BRL", variaveis_brl),
    ("RETORNO_ORIGINAL", variaveis_originais)
]:
    for i, variavel_a in enumerate(variaveis):
        for variavel_b in variaveis[i + 1:]:

            base_par = (
                base_correlacoes
                .select(variavel_a, variavel_b)
                .dropna()
            )

            quantidade = base_par.count()

            correlacao = (
                base_par.stat.corr(
                    variavel_a,
                    variavel_b
                )
                if quantidade >= 3 else None
            )

            pares_correlacao.append({
                "perspectiva": perspectiva,
                "variavel_a": variavel_a,
                "variavel_b": variavel_b,
                "observacoes": quantidade,
                "correlacao_pearson": correlacao
            })

gold_correlacoes = spark.createDataFrame(pares_correlacao)

display(
    gold_correlacoes
    .orderBy(
        "perspectiva",
        F.desc(F.abs(F.col("correlacao_pearson")))
    )
)

## 8. Cenários de mercado

Para analisar o comportamento dos ativos em diferentes condições de mercado, os meses são classificados de acordo com a volatilidade realizada do Ibovespa.

A volatilidade mensal realizada é calculada pelo desvio-padrão dos retornos entre pregões dentro de cada mês. São considerados apenas meses com pelo menos dez retornos válidos.

Os meses são divididos em três grupos de tamanho aproximadamente semelhante: baixa, média e alta volatilidade. A classificação é relativa ao período estudado e não representa uma definição universal de crise ou estabilidade.

Os retornos mensais dos ativos são então comparados entre os cenários, permitindo avaliar diferenças de desempenho e risco de maneira descritiva.

In [0]:
ibov_volatilidade_mensal = (
    gold_precos_diarios
    .filter(F.col("ativo") == "IBOV")
    .filter(F.col("retorno_diario").isNotNull())
    .withColumn(
        "mes_referencia",
        F.trunc("data", "month")
    )
    .groupBy("mes_referencia")
    .agg(
        F.count("retorno_diario").alias("pregoes_com_retorno"),
        F.stddev_samp("retorno_diario").alias(
            "volatilidade_diaria_ibov"
        )
    )
    .filter(F.col("pregoes_com_retorno") >= 10)
    .withColumn(
        "volatilidade_realizada_anualizada",
        F.col("volatilidade_diaria_ibov") *
        F.sqrt(F.lit(252.0))
    )
)

janela_cenarios = Window.orderBy(
    "volatilidade_realizada_anualizada",
    "mes_referencia"
)

gold_cenarios_mensais = (
    ibov_volatilidade_mensal
    .withColumn(
        "grupo_volatilidade",
        F.ntile(3).over(janela_cenarios)
    )
    .withColumn(
        "cenario",
        F.when(
            F.col("grupo_volatilidade") == 1,
            "BAIXA_VOLATILIDADE"
        )
        .when(
            F.col("grupo_volatilidade") == 2,
            "MEDIA_VOLATILIDADE"
        )
        .otherwise("ALTA_VOLATILIDADE")
    )
    .drop("grupo_volatilidade")
)

display(
    gold_cenarios_mensais
    .orderBy("mes_referencia")
)

In [0]:
base_cenarios = (
    base_retornos
    .join(
        gold_cenarios_mensais.select(
            "mes_referencia",
            "cenario",
            "volatilidade_realizada_anualizada"
        ),
        on="mes_referencia",
        how="inner"
    )
)

gold_desempenho_cenarios = (
    base_cenarios
    .groupBy(
        "cenario",
        "ativo",
        "classe_ativo"
    )
    .agg(
        F.count("retorno_brl").alias("meses"),
        F.avg("retorno_brl").alias("retorno_medio_mensal"),
        F.stddev_samp("retorno_brl").alias(
            "volatilidade_mensal"
        ),
        F.avg(
            F.when(
                F.col("retorno_brl") > 0,
                F.lit(1.0)
            ).otherwise(F.lit(0.0))
        ).alias("proporcao_meses_positivos"),
        F.min("retorno_brl").alias("pior_retorno_mensal"),
        F.max("retorno_brl").alias("melhor_retorno_mensal")
    )
)

display(
    gold_desempenho_cenarios
    .orderBy("cenario", "ativo")
)

## 9. Evolução acumulada dos investimentos

A evolução acumulada é representada por um índice de crescimento que parte de 1 no fechamento de janeiro de 2021.

O índice é calculado pela composição dos retornos mensais em reais. Também é construída uma trajetória real, descontada a inflação mês a mês.

A série permite comparar visualmente o crescimento acumulado dos ativos e a diferença entre resultados nominais e reais, sem assumir aportes adicionais, rebalanceamento ou reinvestimento de proventos.

In [0]:
gold_evolucao_acumulada = (
    base_retornos
    .withColumn(
        "indice_nominal",
        F.exp(
            F.sum(
                F.log(1 + F.col("retorno_brl"))
            ).over(janela_acumulada)
        )
    )
    .withColumn(
        "indice_real",
        F.exp(
            F.sum(
                F.log(1 + F.col("retorno_real_mensal"))
            ).over(janela_acumulada)
        )
    )
    .withColumn(
        "retorno_acumulado",
        F.col("indice_nominal") - 1
    )
    .withColumn(
        "retorno_real_acumulado",
        F.col("indice_real") - 1
    )
)

display(
    gold_evolucao_acumulada
    .orderBy("ativo", "mes_referencia")
    .limit(20)
)

## 10. Validação da qualidade analítica

As tabelas Gold são submetidas a verificações de integridade antes da persistência.

As regras incluem unicidade das chaves, ausência de valores nulos obrigatórios, validade dos retornos, consistência dos fatores de inflação e cobertura temporal.

Valores nulos esperados, como o primeiro retorno de cada ativo e a primeira variação mensal do dólar, são preservados. Não são preenchidos artificialmente.

As validações interrompem a execução em caso de inconsistência, evitando que resultados inválidos sejam gravados na camada Gold.

In [0]:
def validar_gold(df, nome, chave, obrigatorias, regras=None):
    print(f"\n=== VALIDAÇÃO: {nome} ===")

    quantidade = df.count()

    if quantidade == 0:
        raise ValueError(f"{nome}: tabela vazia.")

    duplicados = (
        df.groupBy(*chave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicados > 0:
        raise ValueError(
            f"{nome}: {duplicados} chaves duplicadas."
        )

    for coluna in obrigatorias:
        nulos = df.filter(F.col(coluna).isNull()).count()

        if nulos > 0:
            raise ValueError(
                f"{nome}: {nulos} nulos em {coluna}."
            )

    for descricao, condicao in (regras or []):
        invalidos = df.filter(
            F.coalesce(~condicao, F.lit(True))
        ).count()

        if invalidos > 0:
            raise ValueError(
                f"{nome}: {invalidos} registros inválidos "
                f"na regra {descricao}."
            )

    print(f"Registros: {quantidade}")
    print("Chaves duplicadas: 0")
    print("Nulos obrigatórios: 0")
    print("Regras de qualidade: aprovadas")


def retorno_valido(coluna):
    valor = F.col(coluna)

    return (
        valor.isNull() |
        (
            (valor > -1) &
            (~F.isnan(valor)) &
            (valor != F.lit(float("inf"))) &
            (valor != F.lit(float("-inf")))
        )
    )

In [0]:
validar_gold(
    gold_precos_diarios,
    "Preços diários",
    ["data", "ativo"],
    [
        "data", "ativo", "classe_ativo",
        "unidade_preco", "preco_fechamento"
    ],
    [
        (
            "Preço positivo",
            F.col("preco_fechamento") > 0
        ),
        (
            "Retorno diário válido",
            retorno_valido("retorno_diario")
        )
    ]
)

validar_gold(
    gold_mercado_mensal,
    "Mercado mensal",
    ["mes_referencia", "ativo"],
    [
        "mes_referencia", "ativo",
        "data_fechamento", "preco_fechamento"
    ],
    [
        (
            "Preço positivo",
            F.col("preco_fechamento") > 0
        ),
        (
            "Retorno mensal válido",
            retorno_valido("retorno_mensal")
        )
    ]
)

validar_gold(
    gold_macro_mensal,
    "Macroeconomia mensal",
    ["mes_referencia"],
    [
        "mes_referencia", "ipca_pct_mes",
        "usd_brl", "selic_meta_pct_aa",
        "fator_inflacao"
    ],
    [
        (
            "Fator de inflação positivo",
            F.col("fator_inflacao") > 0
        ),
        (
            "Cotação do dólar positiva",
            F.col("usd_brl") > 0
        ),
        (
            "Selic não negativa",
            F.col("selic_meta_pct_aa") >= 0
        )
    ]
)

validar_gold(
    gold_analise_mensal,
    "Análise mensal integrada",
    ["mes_referencia", "ativo"],
    [
        "mes_referencia", "ativo",
        "preco_fechamento", "ipca_pct_mes",
        "usd_brl", "fator_inflacao"
    ],
    [
        (
            "Retorno em reais válido",
            retorno_valido("retorno_brl")
        ),
        (
            "Retorno real válido",
            retorno_valido("retorno_real_mensal")
        )
    ]
)

In [0]:
validar_gold(
    gold_desempenho_ativos,
    "Desempenho dos ativos",
    ["ativo"],
    [
        "ativo", "meses_com_retorno",
        "retorno_acumulado", "retorno_anualizado",
        "volatilidade_anualizada",
        "retorno_real_acumulado", "drawdown_maximo"
    ],
    [
        (
            "Volatilidade não negativa",
            F.col("volatilidade_anualizada") >= 0
        ),
        (
            "Drawdown entre -100% e 0%",
            F.col("drawdown_maximo").between(-1, 0)
        )
    ]
)

validar_gold(
    gold_volatilidade_anual,
    "Volatilidade anual",
    ["ativo", "ano"],
    [
        "ativo", "ano", "meses_com_retorno",
        "volatilidade_anualizada"
    ],
    [
        (
            "Volatilidade não negativa",
            F.col("volatilidade_anualizada") >= 0
        )
    ]
)

validar_gold(
    gold_correlacoes,
    "Correlações",
    ["perspectiva", "variavel_a", "variavel_b"],
    [
        "perspectiva", "variavel_a",
        "variavel_b", "observacoes"
    ],
    [
        (
            "Correlação no intervalo válido",
            F.col("correlacao_pearson").isNull() |
            F.col("correlacao_pearson").between(-1, 1)
        ),
        (
            "Quantidade mínima de observações",
            F.col("observacoes") >= 3
        )
    ]
)

validar_gold(
    gold_cenarios_mensais,
    "Cenários mensais",
    ["mes_referencia"],
    [
        "mes_referencia", "cenario",
        "volatilidade_realizada_anualizada"
    ],
    [
        (
            "Volatilidade não negativa",
            F.col("volatilidade_realizada_anualizada") >= 0
        )
    ]
)

validar_gold(
    gold_desempenho_cenarios,
    "Desempenho por cenário",
    ["cenario", "ativo"],
    [
        "cenario", "ativo", "meses",
        "retorno_medio_mensal",
        "proporcao_meses_positivos"
    ],
    [
        (
            "Proporção entre 0 e 1",
            F.col("proporcao_meses_positivos").between(0, 1)
        )
    ]
)

validar_gold(
    gold_evolucao_acumulada,
    "Evolução acumulada",
    ["mes_referencia", "ativo"],
    [
        "mes_referencia", "ativo",
        "indice_nominal", "indice_real"
    ],
    [
        (
            "Índices de crescimento positivos",
            (F.col("indice_nominal") > 0) &
            (F.col("indice_real") > 0)
        )
    ]
)

## 11. Verificação da cobertura temporal

A cobertura mensal é verificada antes da gravação, considerando os cinco ativos e os 60 meses de observação.

São esperados 59 retornos mensais comparáveis por ativo, de fevereiro de 2021 a dezembro de 2025. Janeiro de 2021 é preservado como referência inicial.

A verificação também confirma que os indicadores macroeconômicos possuem os 60 meses necessários para a integração.

In [0]:
cobertura_mercado = (
    gold_mercado_mensal
    .groupBy("ativo")
    .agg(
        F.count("*").alias("meses_preco"),
        F.count("retorno_mensal").alias("meses_retorno"),
        F.min("mes_referencia").alias("primeiro_mes"),
        F.max("mes_referencia").alias("ultimo_mes")
    )
)

display(cobertura_mercado)

cobertura_macro = (
    gold_macro_mensal
    .agg(
        F.count("*").alias("meses"),
        F.min("mes_referencia").alias("primeiro_mes"),
        F.max("mes_referencia").alias("ultimo_mes")
    )
)

display(cobertura_macro)

falhas_cobertura = (
    cobertura_mercado
    .filter(
        (F.col("meses_preco") != 60) |
        (F.col("meses_retorno") != 59) |
        (F.col("primeiro_mes") != F.lit("2021-01-01").cast("date")) |
        (F.col("ultimo_mes") != F.lit("2025-12-01").cast("date"))
    )
    .count()
)

if falhas_cobertura > 0:
    raise ValueError("Cobertura mensal incompleta para algum ativo.")

if gold_macro_mensal.count() != 60:
    raise ValueError("Cobertura macroeconômica mensal incompleta.")

print("Cobertura temporal validada.")

## 12. Persistência das tabelas Gold

Após a aprovação das validações, as tabelas analíticas são persistidas em formato Delta no schema Gold.

O modo `overwrite` permite a reprodução integral do notebook. As tabelas Bronze e Silver não são modificadas.

A camada Gold contém tabelas de diferentes granularidades, incluindo observações diárias, mensais, resumos por ativo, resumos anuais e matrizes de correlação em formato tabular.

In [0]:
tabelas_gold = {
    "precos_diarios": gold_precos_diarios,
    "mercado_mensal": gold_mercado_mensal,
    "macro_mensal": gold_macro_mensal,
    "analise_mensal": gold_analise_mensal,
    "desempenho_ativos": gold_desempenho_ativos,
    "volatilidade_anual": gold_volatilidade_anual,
    "correlacoes": gold_correlacoes,
    "cenarios_mensais": gold_cenarios_mensais,
    "desempenho_cenarios": gold_desempenho_cenarios,
    "evolucao_acumulada": gold_evolucao_acumulada
}

for nome, df in tabelas_gold.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela("gold", nome))
    )

    print(f"Tabela gravada: gold.{nome}")

## 13. Documentação no Data Catalog

As tabelas Gold recebem descrições de finalidade, granularidade e regras metodológicas.

O catálogo complementa o dicionário de dados do projeto, que documentará os campos, tipos, unidades, fontes, transformações e chaves.

A documentação permite rastrear a origem das análises e compreender as limitações das métricas calculadas.

In [0]:
comentarios_gold = {
    "precos_diarios": (
        "Preços de fechamento e retornos entre observações consecutivas. "
        "Granularidade: ativo e data. Criptomoedas em USDT e Ibovespa em pontos."
    ),
    "mercado_mensal": (
        "Último fechamento disponível por mês e ativo, com retorno mensal. "
        "Janeiro de 2021 é a referência inicial sem retorno."
    ),
    "macro_mensal": (
        "IPCA mensal, PTAX USD/BRL de fim de mês e Meta Selic anual. "
        "Inclui variação cambial, fator de inflação e variação da Selic em pontos percentuais."
    ),
    "analise_mensal": (
        "Integração mensal de mercado e macroeconomia. "
        "Inclui retornos originais, aproximação em reais e retorno real. "
        "A conversão de USDT para BRL assume paridade USDT/USD."
    ),
    "desempenho_ativos": (
        "Resumo de desempenho dos ativos no intervalo comparável. "
        "Inclui retorno acumulado, anualizado, volatilidade, retorno real e drawdown."
    ),
    "volatilidade_anual": (
        "Indicadores anuais de retorno e volatilidade calculados a partir de retornos mensais em reais."
    ),
    "correlacoes": (
        "Correlações de Pearson entre ativos e indicadores macroeconômicos. "
        "Perspectivas original e em reais, com quantidade de observações por par."
    ),
    "cenarios_mensais": (
        "Classificação retrospectiva dos meses em baixa, média e alta volatilidade "
        "com base na volatilidade realizada do Ibovespa."
    ),
    "desempenho_cenarios": (
        "Resumo descritivo do comportamento dos ativos nos cenários de volatilidade do Ibovespa."
    ),
    "evolucao_acumulada": (
        "Trajetórias acumuladas nominais e reais dos ativos, com base inicial no fechamento de janeiro de 2021."
    )
}

for tabela, comentario in comentarios_gold.items():
    spark.sql(
        f"COMMENT ON TABLE {nome_tabela('gold', tabela)} "
        f"IS '{comentario}'"
    )

print("Descrições das tabelas Gold registradas no catálogo.")

## 14. Validação final da persistência

As tabelas são consultadas novamente após a gravação para confirmar a existência dos dados e a correspondência entre os registros persistidos e os DataFrames analíticos.

A validação final também apresenta as quantidades de registros de cada tabela, permitindo conferir a estrutura da camada Gold antes da etapa de análise e elaboração do relatório.

In [0]:
print("=== VALIDAÇÃO FINAL DA CAMADA GOLD ===")

for nome, df_origem in tabelas_gold.items():
    tabela_persistida = spark.table(
        nome_tabela("gold", nome)
    )

    quantidade_origem = df_origem.count()
    quantidade_persistida = tabela_persistida.count()

    print(
        f"{nome}: {quantidade_persistida} registros "
        f"| origem: {quantidade_origem}"
    )

    if quantidade_origem != quantidade_persistida:
        raise ValueError(
            f"Divergência na persistência de gold.{nome}."
        )

print("\nTodas as tabelas Gold foram persistidas com sucesso.")

## Resultado da camada Gold

A camada Gold foi construída a partir das cinco fontes tratadas na Silver, sem necessidade de executar novamente as coletas.

Foram produzidas tabelas de preços e retornos diários, fechamentos mensais, indicadores macroeconômicos, análise integrada, desempenho, volatilidade, correlações, cenários de mercado e evolução acumulada.

As validações confirmam a integridade das chaves, a consistência dos valores, a cobertura temporal e a persistência das tabelas Delta.

As análises preservam as distinções entre USDT, USD e BRL, as diferentes frequências dos indicadores e as limitações das correlações e dos cenários retrospectivos.

A próxima etapa consiste em consultar as tabelas Gold para responder às perguntas de negócio, produzir visualizações e elaborar a discussão crítica dos resultados.